In [ ]:
from pathlib import Path
import json
import re
import uuid
import logging
from datetime import datetime
from dotenv import load_dotenv
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
load_dotenv()

True

## Evaluation pipeline

### Define helper functions

In [ ]:
def run_format_check(dataset_name):
    path = f'.datasets/{dataset_name}/'
    for file in Path(path + "/01_fabricated").rglob('*'):
        if file.is_file():
            pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
            match = pattern.search(file.stem)
            if not match:
                print(f"Filename {file.stem} does not match the expected pattern.")

def load_data(dataset_name):
    dataset_path = f'.datasets/{dataset_name}/dataset_{dataset_name}.json'
    try:
        with open(dataset_path, 'r') as f:
            dataset = json.load(f)
        return dataset
    except FileNotFoundError:
        print(f"Dataset file {dataset_path} not found.")
        return None
    
def save_results(dataset_name : str, data : dict, llm_model : str):
    folder_path = Path(f'.datasets/{dataset_name}/results/')
    folder_path.mkdir(parents=True, exist_ok=True)
    output_path = folder_path / f'{llm_model}_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.json'
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    logger.info(f"Results saved to {output_path}")

def get_attachments(dataset_name, start_date = None, end_date = None,):

    folder_path = Path(f'./{dataset_name}/01_data/')
    attachments = []
    for file in Path(folder_path).glob('*'):
        if file.is_file() and file.suffix in ['.pdf', '.docx', '.xlsx', '.csv', '.txt','.md','.eml']:
            if start_date is None and end_date is None:
                logger.debug(f'No date range specified, adding file {file.name} to attachments.')
                attachments.append(str(folder_path / file.name))
                continue
            
            pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
            match = pattern.search(file.stem)
            if match:
                file_date = match.group(1)
                if start_date and end_date:
                    if start_date <= file_date <= end_date:
                        logger.debug(f"File {file.name} is within the date range {start_date} to {end_date}.")
                        attachments.append(str(folder_path / file.name))
                elif start_date:
                    if file_date >= start_date:
                        logger.debug(f"File {file.name} is after the start date {start_date}.")
                        attachments.append(str(folder_path / file.name))
                elif end_date:
                    if file_date <= end_date:
                        logger.debug(f"File {file.name} is before the end date {end_date}.")
                        attachments.append(str(folder_path / file.name))
            else:
                logger.warning(f"Filename {file.stem} does not contain a valid date.")

    if not attachments:
        logger.warning("No attachments found within the specified date range.")
    return attachments

### Prepare data

In [ ]:
dataset_name = "THRD-2021-163881"
path = f'.datasets/{dataset_name}/'
data = load_data(dataset_name)
all_attachments = set()
total = 0
prev_date = None

for idx, session in enumerate(data.get("sessions", [])):
    current_date = session["date"]
    
    attachments = get_attachments(
        dataset_name,
        start_date=prev_date,
        end_date=current_date   
    )
    non_duplicate = set()
    added_count = 0
    for att in attachments:
        if att not in all_attachments:
            all_attachments.add(att)
            non_duplicate.add(att)
            added_count += 1
        else:
            logger.debug(f"Attachment {att} already exists in the set of all attachments.")
    session["attachments"] = list(non_duplicate)    
    total += added_count
    logger.info(f"Session {idx}: extracted {len(attachments)} files, added {added_count} new ones "
                f"(range: {prev_date or '–'} → {current_date or '–'})")
    prev_date = current_date

if data.get("sessions"):
    last_attachments = get_attachments(dataset_name, start_date=data.get("sessions", [])[-1]["date"])
    added_count = 0
    for att in last_attachments:
        if att not in all_attachments:
            all_attachments.add(att)
            added_count += 1
    total += added_count
    logger.info(f"After last session: extracted {len(last_attachments)} files, added {added_count} new ones "
                f"(range: {data.get('sessions', [])[-1]['date']} → –)") 


DEBUG:__main__:File 2007-03-22_38_byggetillatelse_garasje_carport_2007-03-22.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-07-28_04_epost_2019-07-28_utbedring_bekreftelse.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-05-13_02_epost_2019-05-13_selgers_svar.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2020-03-12_07_rapport_2020-03-12_byggesoek_betongdekke.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-08-18_05_epost_2019-08-18_ny_lekkasje.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-09-10_06a_skaderapport_2019-09-10_K2.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2020-01-28_leieavtale_2020_tilbygg.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-06-01_00_kjøpskontrakt_2019-06-01.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-06-30_85_kvitteringer_flyttekostnader_2019-06-30.txt is before the end date 2020-03-15.
DEBUG:__main__:File 2019-05-15_00b_salgsoppgave

### GATHER RESULTS

In [4]:
from agent.agent import Agent
from models import AskAgentRequest, AttachmentModel
from agent.utils import PROMPT
from agent.tools import TOOLS
import os
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg_pool import AsyncConnectionPool

async def init_agent():
    connection_string = os.getenv("SUPABASE_DB_URL")

    pool = AsyncConnectionPool(conninfo=connection_string, open=False)
    await pool.open()

    checkpointer = AsyncPostgresSaver(pool)

    agent = Agent(
        tools=TOOLS,
        prompt=PROMPT,
        checkpointer=checkpointer,
    )
    logger.info("Agent initialized with AsyncPostgresSaver checkpointer")
    return agent

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [ ]:
from base64 import b64encode
llm_model = "google_gemini-2.5-pro"
logger.info(f"=========== STARTING EVALUATION ===========")
logger.info(f'Dataset name : {dataset_name} | Total sessions: {len(data.get("sessions", []))} | Project ID: {data.get("project_id", "Unknown")} | User ID: {data.get("user_id", "Unknown")}')
logger.info(f'===========================================')

file_type_map = {"txt" : "text/plain", "pdf": "application/pdf", "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                 "xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", "csv": "text/csv", "md": "text/markdown", "eml": "message/rfc822"}
data["llm_model"] = llm_model
agent_class = await init_agent()
for idx, session in enumerate(data.get("sessions", [])):
    if idx < 1: #FOR TESTING
        #====================
        # INIT OR UPDATE
        #====================
        logger.info(f"Session date: {session['date']}, session name: {session['session_name']} attachments: {len(session['attachments'])}")
        attachments = []
        query_id = str(uuid.uuid4())
        for i, att in enumerate(session["attachments"]):
            #if i < 3: #FOR TESTING
            with open(att, 'rb') as f:
                content = b64encode(f.read()).decode('utf-8')
            file_id = str(uuid.uuid4())
            attachment_obj = AttachmentModel(
                filename=Path(att).name,
                file_type = file_type_map.get(Path(att).suffix.lstrip('.'), None),
                content=content,
                file_id = file_id,
                path = f"{data.get('user_id')}/{session['session_id']}/{file_id}.{Path(att).suffix.lstrip('.')}",
                size = len(content),
                query_id = query_id
            )
            attachments.append(attachment_obj)

        input_obj = AskAgentRequest(
                                    question=session.get("init_query"),
                                    session_id = session.get("session_id"),
                                    llm_model=llm_model,
                                    query_id = query_id,
                                    project_id=data.get("project_id"),
                                    attachments=attachments
                                    )
        if idx == 0:
            async for response in agent_class.initialize_project(query = input_obj,
                                                    user_id = data.get("user_id"),
                                                    ):
                logger.debug(f"Received response during initialization: {response}")
        else:
            async for response in agent_class.update_project(query=input_obj,
                                                        user_id = data.get("user_id"),):
                    logger.debug(f"Received response during update: {response}")
        
        #====================
        # RUN CONVERSATION
        #====================
        
        for conv in session["conversation"]:
            input_obj = AskAgentRequest(
                                        question=conv.get("input"),
                                        session_id = session.get("session_id"),
                                        llm_model="google_gemini-2.5-pro",
                                        query_id = query_id,
                                        project_id=data.get("project_id"),
                                        attachments=[],
                                        
                                        )
            async for response in agent_class.stream_response(query = input_obj,
                                                    user_id = data.get("user_id"),
                                                    ):
                if response.get("type") == "ai":
                    ai_response = response
                    answer = response.get("data", {}).get("token_stream", "No content")
                    logger.info(f"Received answer: {answer}")
            
            conv["model_response"] = answer


INFO:__main__:=========== STARTING EVALUATION ===========
INFO:__main__:Dataset name : THRD-2021-163881 | Total sessions: 10 | Project ID: ee9ee007-92f7-4fdc-ba5a-d7cb55694241 | User ID: 53d63d18-cfa1-416e-96e8-770c8f66507b
INFO:__main__:===========================================
INFO:__main__:Agent initialized with AsyncPostgresSaver checkpointer
INFO:__main__:Session date: 2020-03-15, session name: Prosjekt-initialisering attachments: 18
DEBUG:__main__:Received response during initialization: {'type': 'status', 'phase': ['parse-documents'], 'status': 'starting', 'data': {'attachments': 3}, 'timestamp': '2026-02-23T16:31:56.544760', 'query_id': '8519f395-8716-4424-902a-d309a8a083c4'}
DEBUG:__main__:Received response during initialization: {'type': 'status', 'phase': ['parse_doc'], 'status': 'starting', 'data': {'filename': '2019-05-13_02_epost_2019-05-13_selgers_svar.txt', 'file_id': '36dce6b2-ca5c-4ada-ac7c-77ff11134411', 'progress': 0, 'total': 3}, 'timestamp': '2026-02-23T16:31:56

Storage endpoint URL should have a trailing slash.


INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/36dce6b2-ca5c-4ada-ac7c-77ff11134411.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/8a0e3141-01e9-4d9d-8ccf-c93b0ca4746a.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/5863698c-29ed-4b7d-bbe5-dcccdebc8469.txt
INFO:database.storage_modules:Upload complete: 3 succeeded, 0 failed out of 3 total
DEBUG:__main__:Received response during initialization: {'type': 'status', 'phase': ['storage'], 'status': 'complete', 'data': {'progress': 1, 'total': 5, 'storage_type': ['file_storage']}, 'timestamp': '2026-02-23T16:31:57.412906', 'query_id': '8519f395-8716-4424-902a-d309a8a083c4'}
DEBUG:__main__:Received response during initialization: {'type': 'status', '

In [6]:
save_results(dataset_name, data)

INFO:__main__:Results saved to ./THRD-2021-163881/results_THRD-2021-163881.json


### EVALUATE RESULTS

In [7]:
import deepeval